# Business Entity Resolution: full pipeline (Amazon ML Challenge 2026)

End to end, one notebook: **prep → blocking → e5 embeddings (GPU) → stage-1 pruning (GPU) → features → XGBoost (GPU) → test inference → decision tuning → validator**.
Code: https://github.com/hazelmayank/ml-challenge (`code/business_entity_resolution/src/`).

**Settings before running:** Accelerator **GPU T4 ×2**, Internet **On**, Input: the challenge dataset (a folder containing `train/` and `test/` TSVs, found automatically under `/kaggle/input`).
Run with **Save Version → Save & Run All (Commit)** (~4.5 h, runs unattended), or cell by cell.

Outputs (in `/kaggle/working`):
- `output/matching_results.tsv`: tuned decision rule, **the file to upload**
- `output/candidate_pairs.tsv`: the exact candidate set the model scored
- `output_base/matching_results.tsv`: plain threshold rule (for comparison)
- `artifacts/`: models, decision files, all logs, run summary

## 1. Configuration

In [ ]:
RUN_NAME = 'v4'          # label for logs/artifacts
COMMIT   = None          # None = latest main; or a commit hash to reproduce an exact run
MODEL    = 'xgb'         # 'xgb' (XGBoost on GPU) or 'lgb' (LightGBM on CPU)
DO_TRAIN, DO_TEST, DO_DECIDE, DO_REVIEW = True, True, True, True

WORK   = '/tmp/work'                    # large intermediate files (1 TB disk, not saved)
OUT    = '/kaggle/working/output'       # final submission files (saved with the version)
OUT_BASE = '/kaggle/working/output_base'
ART    = '/kaggle/working/artifacts'
REPO_URL = 'https://github.com/hazelmayank/ml-challenge.git'

## 2. Code, data paths, environment

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
from pathlib import Path

REPO = Path('/kaggle/working/ml-challenge')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if COMMIT:
    subprocess.run(['git', '-C', str(REPO), 'checkout', COMMIT], check=True)
HEAD = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
SRC = REPO / 'code' / 'business_entity_resolution' / 'src'
VALIDATOR = REPO / 'student_resource' / 'utils' / 'validate_submission.py'

hits = glob.glob('/kaggle/input/**/train/train_source1.tsv', recursive=True)
assert hits, 'Dataset not found under /kaggle/input: add the challenge dataset as input'
DATA = str(Path(hits[0]).parents[1])

os.environ.update(BER_DATA=DATA, BER_WORK=WORK, BER_OUTPUT=OUT, BER_MODEL=MODEL)
for p in (WORK, f'{WORK}/logs', OUT, OUT_BASE, ART):
    Path(p).mkdir(parents=True, exist_ok=True)
shutil.copy(REPO / 'code/business_entity_resolution/artifacts/indic_dict.json', WORK)  # learned from train pairs
print('commit', HEAD, '| data', DATA, os.listdir(DATA))

## 3. Dependencies and hardware check

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO / 'code/business_entity_resolution/requirements.txt')], check=True)
subprocess.run('nproc; free -g | head -2; df -h /kaggle/working /tmp | tail -2; nvidia-smi -L', shell=True)
check = '''
import torch, xgboost, lightgbm, polars, numpy
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(),
      "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__,
      "| polars", polars.__version__, "| numpy", numpy.__version__)
'''
subprocess.run([sys.executable, '-c', check], check=True)

## 4. Helpers

In [ ]:
def step(*args, log=None):
    """Run a pipeline script from src/ as a subprocess; stream and log its output."""
    name = log or '_'.join(str(a).replace('.py', '') for a in args) + '.log'
    logf = Path(os.environ['BER_WORK']) / 'logs' / name
    t = time.time()
    with open(logf, 'a') as f:
        f.write(f'# {RUN_NAME} commit {HEAD} {time.ctime()} {args}\n')
        p = subprocess.Popen([sys.executable, '-u', *map(str, args)], cwd=SRC, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
        for line in p.stdout:
            print(line, end='', flush=True); f.write(line); f.flush()
        rc = p.wait()
    print(f'--> {args[0]} exit {rc} in {time.time() - t:.0f}s', flush=True)
    if rc:
        raise RuntimeError(f'{args} failed ({rc}); see {logf}')

def fresh(*names):
    for n in names:
        shutil.rmtree(Path(os.environ['BER_WORK']) / n, ignore_errors=True)

def disk():
    subprocess.run(f'df -h /kaggle/working /tmp | tail -2; du -sh {WORK} /kaggle/working/* 2>/dev/null', shell=True)

## 5. Preprocessing (normalisation) — train and test

In [ ]:
step('prep.py', 'train', 'test')      # ~10 min

## 6. Training pipeline (all train records)

In [ ]:
if DO_TRAIN:
    step('block.py', 'train');  fresh('train_cand_parts')        # token blocking, ~25 min
    step('embed.py', 'train');  fresh('train_cand_emb_parts')    # e5 kNN candidates (GPU), ~35 min
    step('prune.py', 'train')                                    # stage-1 model + pruning (GPU), ~10 min
    step('features.py', 'train_cand')                            # pair features, ~10 min
    step('train.py', 1.0)                                        # 2-fold CV + final model (GPU), ~15 min
    disk()

## 7. Test inference

In [ ]:
if DO_TEST:
    step('block.py', 'test');   fresh('test_cand_parts')         # ~25 min
    step('embed.py', 'test');   fresh('test_cand_emb_parts')     # ~35 min
    step('prune.py', 'test')
    step('features.py', 'test_cand')
    step('predict.py')                                           # base rule -> OUT
    for f in ('matching_results.tsv', 'candidate_pairs.tsv'):
        shutil.copy(Path(OUT) / f, Path(OUT_BASE) / f)
    disk()

## 8. Decision tuning (on out-of-fold scores) and validation

In [ ]:
if DO_DECIDE:
    step('decide.py')                                            # rewrites OUT/matching_results.tsv
for out in (OUT_BASE, OUT):
    step(VALIDATOR, '--matching', f'{out}/matching_results.tsv', '--candidate', f'{out}/candidate_pairs.tsv',
         '--test-dir', f'{DATA}/test', log=f'validate_{Path(out).name}.log')

## 9. France review (no labels: eyeball accepted / borderline pairs)

In [ ]:
if DO_REVIEW:
    step('review.py', 'France', '25', log='review_france.log')

## 10. Summary and artifacts

In [ ]:
import polars as pl
W = Path(WORK)
summary = {'run': RUN_NAME, 'commit': HEAD, 'model': MODEL}
for f in ('decision.json', 'decision_tuned.json', 'prune.json'):
    if (W / f).exists():
        d = json.loads((W / f).read_text()); d.pop('features', None); summary[f] = d
for out in (OUT_BASE, OUT):
    for kind in ('matching_results', 'candidate_pairs'):
        col = 'matched_entity_ids' if kind == 'matching_results' else 'candidate_entity_ids'
        df = pl.read_csv(f'{out}/{kind}.tsv', separator='\t', infer_schema_length=0,
                         missing_utf8_is_empty_string=True, quote_char=None)
        n = df.select(pl.when(pl.col(col) == '').then(0)
                      .otherwise(pl.col(col).str.split(',').list.len()).alias('n'))['n']
        summary[f'{Path(out).name}/{kind}'] = {'rows': len(df), 'ids': int(n.sum()),
                                               'per_S1_mean': round(float(n.mean()), 2),
                                               'empty_rows': int((n == 0).sum())}
print(json.dumps(summary, indent=1))
(Path(ART) / 'summary.json').write_text(json.dumps(summary, indent=1))
for f in ('model.xgb.ubj', 'model.txt', 'prune.xgb.ubj', 'prune.txt', 'decision.json',
          'decision_tuned.json', 'prune.json', 'indic_dict.json'):
    if (W / f).exists(): shutil.copy(W / f, ART)
shutil.copytree(W / 'logs', Path(ART) / 'logs', dirs_exist_ok=True)
disk()